# Lesson 24: Large Language Models — parameters and scaling laws 📈
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain what makes a language model "large", and the three stages that turn it into a chatbot
2. Understand both kinds of **LLM parameters**: the **weights** inside the model, and the **generation settings** you choose (temperature, top-k, top-p, max tokens, repetition penalty)
3. Run a mini **scaling-law** experiment and use the "Chinchilla" rule of thumb

**⏱️ Timing:** what is an LLM 10' | weights & memory 10' | generation settings 20' | scaling laws 20'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🏗️ What is a Large Language Model?
An **LLM** is a GPT-style Transformer (Lessons 7, 8, 23) that is **large** in three ways:
| | Example of scale |
|---|---|
| **Parameters** (weights) | billions to trillions of numbers |
| **Training data** | trillions of tokens of text (and code, images…) |
| **Compute** | thousands of GPUs running for weeks or months |

### 🍳 From raw model to chatbot — three stages
1. **Pretraining** — predict the next token on a huge pile of text → a **base model** that can continue any text (like GPT-2 in Lesson 23)
2. **Instruction tuning (SFT)** — train on examples of questions and good answers → it learns to **follow requests**
3. **Preference tuning (e.g. RLHF)** — humans rank answers; the model is nudged toward the preferred ones → more **helpful, honest and safe** (Lesson 27)

> Analogy: pretraining = reading the whole library; instruction tuning = an apprenticeship with a mentor; RLHF = getting feedback from customers.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
llm = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype="auto", device_map="auto")
print("✅ loaded", NAME)

## Part 1: Parameters meaning #1 — the weights 🔢
"0.5B" in the name means about **0.5 billion parameters** (weights). Let's count them and see where they live.

In [ ]:
total = sum(p.numel() for p in llm.parameters())
print(f"Total parameters: {total:,}")
groups = {}
for name, p in llm.named_parameters():
    key = "embeddings" if "embed" in name else "attention" if "attn" in name else "feed-forward (MLP)" if "mlp" in name else "other"
    groups[key] = groups.get(key, 0) + p.numel()
for k, v in groups.items():
    print(f"  {k:20} {v / total:5.1%}")
print("\nConfig:", {k: getattr(llm.config, k) for k in ["num_hidden_layers", "hidden_size", "num_attention_heads", "vocab_size", "max_position_embeddings"]})

In [ ]:
# How much memory does a model need just to LOAD it? parameters × bytes per parameter
def memory_gb(params_billions, bits):
    return params_billions * 1e9 * bits / 8 / 1e9

print(f"{'model size':>10} | {'32-bit':>7} | {'16-bit':>7} | {'4-bit':>6}")
for b in [0.5, 3, 7, 70, 400]:
    print(f"{b:>9}B | {memory_gb(b, 32):6.0f}GB | {memory_gb(b, 16):6.0f}GB | {memory_gb(b, 4):5.1f}GB")
print("\nA free Colab T4 GPU has about 15 GB.")

### 🧩 Exercise 1
1. Which is the biggest model you could load on a T4 in 16-bit? In 4-bit? (4-bit "quantisation" is how Lesson 26's QLoRA fits big models on small GPUs.)
2. Training needs roughly **4× more** memory than just loading (gradients and optimiser state). Why can't schools train big LLMs?

## Part 2: Parameters meaning #2 — generation settings 🎛️
When you call an LLM API you'll see settings like `temperature`, `top_p`, `max_tokens`. Let's look at the model's **next-token probabilities** and see what each setting does to them.

In [ ]:
import matplotlib.pyplot as plt

prompt = "My favourite school subject is"
ids = tok(prompt, return_tensors="pt").to(llm.device)
with torch.no_grad():
    logits = llm(**ids).logits[0, -1].float()

def show(probs, title, k=12):
    top = torch.topk(probs, k)
    labels = [repr(tok.decode(i)) for i in top.indices]
    plt.figure(figsize=(10, 2.5)); plt.bar(labels, top.values.cpu()); plt.xticks(rotation=45); plt.title(title); plt.show()

for t in [0.2, 1.0, 2.0]:
    show(torch.softmax(logits / t, dim=0), f"temperature = {t}")

In [ ]:
def top_k_filter(probs, k):
    keep = torch.topk(probs, k).indices
    out = torch.zeros_like(probs); out[keep] = probs[keep]
    return out / out.sum()

def top_p_filter(probs, p):
    sorted_p, idx = torch.sort(probs, descending=True)
    cut = (torch.cumsum(sorted_p, 0) - sorted_p) < p        # keep the smallest set that adds up to p
    out = torch.zeros_like(probs); out[idx[cut]] = probs[idx[cut]]
    return out / out.sum()

probs = torch.softmax(logits, dim=0)
show(top_k_filter(probs, 5), "top-k = 5: only the 5 most likely tokens can be picked")
show(top_p_filter(probs, 0.5), "top-p = 0.5: the fewest tokens that together cover 50% probability")
print("top-p = 0.5 kept", int((top_p_filter(probs, 0.5) > 0).sum()), "tokens")

In [ ]:
def generate(prompt, **settings):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, **settings)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

q = "Invent a name for a new ice-cream flavour and describe it in one sentence."
experiments = {
    "greedy (no sampling)":           dict(do_sample=False, max_new_tokens=40),
    "temperature 0.3":                dict(do_sample=True, temperature=0.3, max_new_tokens=40),
    "temperature 1.5":                dict(do_sample=True, temperature=1.5, max_new_tokens=40),
    "temperature 1.5 + top-p 0.9":    dict(do_sample=True, temperature=1.5, top_p=0.9, max_new_tokens=40),
    "max_new_tokens = 8":             dict(do_sample=True, temperature=0.7, max_new_tokens=8),
    "repetition_penalty 1.5":         dict(do_sample=True, temperature=0.7, repetition_penalty=1.5, max_new_tokens=40),
}
for name, s in experiments.items():
    print(f"== {name}\n   {generate(q, **s)}\n")

### 🧩 Exercise 2: Pick the settings
| Use case | temperature | top-p | Why? |
|---|---|---|---|
| Maths homework checker | | | |
| Brainstorming band names | | | |
| Translating a school notice | | | |
| Writing a poem | | | |

Also: what happens to `max_new_tokens = 8`? (The answer is cut off mid-sentence. APIs call this `max_tokens`, and you pay per token!)

> Another setting: the **context window** — the maximum number of tokens the model can read at once. This model: `llm.config.max_position_embeddings`. Big commercial models can read hundreds of thousands of tokens.

## Part 3: Scaling laws 📈
In 2020, researchers found something surprising: an LLM's loss drops **smoothly and predictably** as you increase **model size**, **data** and **compute** — following a **power law** (a straight line on a log-log plot). That predictability is why companies invested billions in bigger models.

Let's see it in miniature: train networks of different sizes on MNIST with the same data and steps.

In [ ]:
import torch.nn as nn, numpy as np
from torchvision import datasets, transforms
dev = "cuda" if torch.cuda.is_available() else "cpu"
mnist = datasets.MNIST(".", train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(".", train=False, download=True, transform=transforms.ToTensor())
Xtr = mnist.data[:20000].float().view(-1, 784).to(dev) / 255; ytr = mnist.targets[:20000].to(dev)
Xte = test.data.float().view(-1, 784).to(dev) / 255; yte = test.targets.to(dev)

results = []
for width in [2, 4, 8, 16, 32, 64, 128, 256]:
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(784, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 10)).to(dev)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for step in range(1500):
        idx = torch.randint(0, len(Xtr), (128,), device=dev)
        loss = nn.functional.cross_entropy(net(Xtr[idx]), ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        test_loss = nn.functional.cross_entropy(net(Xte), yte).item()
    n = sum(p.numel() for p in net.parameters())
    results.append((n, test_loss)); print(f"{n:8,} parameters → test loss {test_loss:.3f}")

In [ ]:
N, L = np.array(results).T
slope, intercept = np.polyfit(np.log(N), np.log(L), 1)
plt.loglog(N, L, "o", label="our networks")
plt.loglog(N, np.exp(intercept) * N ** slope, "--", label=f"power law: loss ∝ N^{slope:.2f}")
plt.xlabel("parameters (log scale)"); plt.ylabel("test loss (log scale)"); plt.legend(); plt.show()

Our tiny experiment is noisy and flattens out at the end (the task gets "too easy" and data becomes the limit), but the middle looks like a straight line — a power law!

### 🐭 The Chinchilla rule of thumb
In 2022, DeepMind's **Chinchilla** paper found many LLMs were **too big for the amount of data** they'd seen. Rule of thumb for compute-efficient training: about **20 training tokens per parameter**. Training compute ≈ **6 × parameters × tokens** (in floating-point operations, FLOPs).

In [ ]:
def chinchilla(params_billions):
    tokens = 20 * params_billions * 1e9
    flops = 6 * params_billions * 1e9 * tokens
    gpu_days = flops / (300e12 * 0.4) / 86400          # a modern GPU ≈ 300 TFLOPS, ~40% used efficiently
    print(f"{params_billions:>5}B params → {tokens / 1e12:6.2f} trillion tokens, {flops:.1e} FLOPs, ≈ {gpu_days:,.0f} GPU-days")

for b in [0.5, 7, 70, 400]:
    chinchilla(b)

### 🧩 Exercise 3
1. How many GPU-days for a 70B model? If you had 1,000 GPUs, how many days is that?
2. Scaling laws predict **loss**, not specific skills. Some abilities seem to appear suddenly at larger sizes ("emergent abilities") — though researchers debate whether that's real or an effect of how we measure. Why does it matter to predict this?
3. Is "bigger is better" the whole story? Think about cost, energy use, speed, phones, and smaller models trained longer on better data.

## 📌 Summary
- LLM = a huge GPT-style model: **pretraining → instruction tuning → preference tuning**
- **Parameters** = weights (memory ≈ params × bytes) **and** generation settings (temperature, top-k, top-p, max tokens, repetition penalty)
- **Scaling laws**: loss falls as a smooth power law with model size, data and compute; Chinchilla ≈ 20 tokens per parameter